# targeted dir.
```
env=dev/project=gs25-sales-forecast/.../run_id=20260228T143022Z_a7f3c8d1/
├── metadata/
│   └── run_manifest.yml
├── config/
│   └── config.yml
├── data_refs/
│   └── input_data_ref.yml
├── artifacts/
│   ├── model/
│   │   └── .placeholder
│   ├── metrics/
│   │   └── model_metrics.yml
│   ├── charts/
│   │   └── .placeholder
│   └── explainability/
│       └── .placeholder
└── reports/
    └── report_request.yml
```

# 라이브러리 셋팅

In [1]:
import sys
import os
from pathlib import Path


def inject_project_root() -> None:
    """
    현재 실행 컨텍스트의 절대 경로를 식별하여 sys.path의 최상단에 주입합니다.
    Jupyter 환경(노트북)과 일반 스크립트 실행 모두에 대응합니다.
    """
    try:
        # 일반 파이썬 스크립트 환경
        current_path = Path(__file__).resolve().parent
    except NameError:
        # Jupyter Notebook 환경
        current_path = Path(os.path.abspath('')).resolve()
        
    project_root = str(current_path)
    print(project_root)
    
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

def verify_module_loading() -> None:
    """
    필수 모듈 로드 실패 시 상태 관찰을 위한 로그를 출력하고 즉시 중단(Fail-fast)합니다.
    """
    try:
        import run_pm_utils
        import conf
        print("[Observability] Target modules (run_pm_utils, conf) loaded successfully.")
    except ModuleNotFoundError as e:
        print(f"[Fatal] Failed to load modules. Current sys.path: \n{sys.path}")
        # 디버깅을 위해 현재 디렉토리의 파일 목록을 출력
        print(f"[Fatal] Files in {os.getcwd()}: \n{os.listdir(os.getcwd())}")
        raise e

inject_project_root()
print('-'*30)
verify_module_loading()

/home/ec2-user/SageMaker/gs-ds-env/tests/titanic/notebooks/sean/run_pm
------------------------------
AWS Account ID: 155954279556
AWS Region: us-east-1
[Observability] Target modules (run_pm_utils, conf) loaded successfully.


In [2]:
import os
import time
import json
import shutil
import pickle
import traceback
import argparse
import warnings
import boto3
import papermill as pm
from papermill.exceptions import PapermillExecutionError
import pprint

import run_pm_utils as utils
import conf

In [3]:
pp = pprint.PrettyPrinter(width=41, compact=True, indent=4)
warnings.filterwarnings('ignore')

logs = []

# 하이퍼파라미터 -> argparser 로 받자

In [4]:
# ----------------------------
# Argument Parsing
# ----------------------------
def parse_args():
    try:
        parser = argparse.ArgumentParser(description="AutoML Experiment Configuration")
        parser.add_argument('--project_hashkey', type=str, default='')
        parser.add_argument('--experiment_hashkey', type=str, default='')
        parser.add_argument('--profile_hashkey', type=str, default='')
        parser.add_argument('--experiment_table_name', type=str, default='')
        parser.add_argument('--experiment_result_table_name', type=str, default='')
        parser.add_argument('--dataset_table_name', type=str, default='')
        parser.add_argument('--dataset_profile_table_name', type=str, default='')
        parser.add_argument('--model_repo_table_name', type=str, default='')
        parser.add_argument('--username', type=str, default='')
        parser.add_argument('--task_token', type=str, default='')
        parser.add_argument('--dryrun', type=str, default='false')
        parser.add_argument('--job_type', type=str, default='')
        args, unknown = parser.parse_known_args()
    
        print('args ++++')
        pp.pprint(vars(args))
        print('unknown ++++')
        pp.pprint(unknown)
        return args
    except Exception as e:
        pp.pprint(e)
        logs.append(str(e))
        pass

In [7]:
def prepare_directories(root, job_type):
    try:
        # input
        input_dir = os.path.join(root, job_type, 'input')
        os.makedirs(os.path.join(input_dir, 'profile'), exist_ok=True)
        os.makedirs(os.path.join(input_dir, 'data_refs'), exist_ok=True) # input_data_ref.yml
        os.makedirs(os.path.join(input_dir, 'metadata'), exist_ok=True) # run_manifest.yml
        os.makedirs(os.path.join(input_dir, 'config'), exist_ok=True) # config.yml
        # output
        artifacts_dir = os.path.join(root, job_type, 'artifacts')
        os.makedirs(os.path.join(root, job_type, 'artifacts'), exist_ok=True)
        os.makedirs(os.path.join(artifacts_dir, 'model'), exist_ok=True)
        os.makedirs(os.path.join(artifacts_dir, 'metrics'), exist_ok=True)
        os.makedirs(os.path.join(artifacts_dir, 'charts'), exist_ok=True)
        os.makedirs(os.path.join(artifacts_dir, 'explainability'), exist_ok=True)
        os.makedirs(os.path.join(artifacts_dir, 'reports'), exist_ok=True)
        return input_dir, artifacts_dir
    except Exception as e:
        pp.pprint(e)
        logs.append(str(e))
        pass

## 결과물 디렉토리 만들기

In [17]:
root = str(Path().resolve())
job_type = 'training'

In [18]:
root

'/home/ec2-user/SageMaker/gs-ds-env/tests/titanic/notebooks/sean/run_pm'

In [19]:
prepare_directories(root, job_type)

('/home/ec2-user/SageMaker/gs-ds-env/tests/titanic/notebooks/sean/run_pm/training/input',
 '/home/ec2-user/SageMaker/gs-ds-env/tests/titanic/notebooks/sean/run_pm/training/artifacts')

# S3 로 부터, input 디렉토리 채우기